In [ ]:
# Gerekli kütüphaneleri sessiz yükle
!pip install -q imageio gymnasium stable-baselines3

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import Rectangle
import random
import gymnasium as gym
from gymnasium import spaces
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
import imageio
from tqdm import tqdm

# ============================================
# 10x10 GRID, 11 KONTEYNER, HEPSİ SİPARİŞ (ÜST ÜSTE İSTİF)
# ============================================
class ContainerYardEnv(gym.Env):
    def __init__(self):
        super().__init__()

        self.rows = 10
        self.cols = 10
        self.max_steps = 1000

        self.action_space = spaces.Discrete(6)
        # State: crane(2) + picked(11) + stacking(16) + orders(11) + carrying(1) + target(3) + next_pickup(2) + priority(1) = 47
        self.state_size = 2 + 11 + 16 + 11 + 1 + 3 + 2 + 1
        self.observation_space = spaces.Box(low=0, high=1, shape=(self.state_size,), dtype=np.float32)

        # Kullanıcının seçtiği 11 konteyner
        raw_containers = {
            0: {"pickup": (0,4), "shape": (1,2), "priority": 0},
            1: {"pickup": (2,4), "shape": (1,3), "priority": 0},
            2: {"pickup": (7,4), "shape": (2,1), "priority": 0},
            3: {"pickup": (3,8), "shape": (2,1), "priority": 0},
            4: {"pickup": (1,7), "shape": (1,2), "priority": 1},
            5: {"pickup": (5,7), "shape": (1,3), "priority": 1},
            6: {"pickup": (2,9), "shape": (2,1), "priority": 1},
            7: {"pickup": (6,7), "shape": (3,1), "priority": 1},
            8: {"pickup": (3,5), "shape": (1,3), "priority": 2},
            9: {"pickup": (5,5), "shape": (2,1), "priority": 2},
            10: {"pickup": (0,6), "shape": (2,1), "priority": 2}
        }

        self.containers_original = {}
        for cid, data in raw_containers.items():
            r, c = data['pickup']
            rows, cols = data['shape']
            cells = [(r + i, c + j) for i in range(rows) for j in range(cols)]
            priority = data['priority']
            color = 'green' if priority == 0 else ('yellow' if priority == 1 else 'red')
            self.containers_original[cid] = {
                'cells': cells,
                'size': (rows, cols),
                'name': f'K{cid}',
                'pickup': data['pickup'],
                'priority': priority,
                'color': color
            }

        self.blocked_cells = []
        for r in range(3):
            for c in range(4):
                self.blocked_cells.append((r, c))
        for r in range(7, 10):
            for c in range(4):
                self.blocked_cells.append((r, c))

        self.stacking_width = 4
        self.stacking_height = 4
        self.reset()

    def reset(self, seed=None):
        super().reset(seed=seed)
        all_ids = list(self.containers_original.keys())
        # Tüm konteynerler sipariş (11 adet)
        self.orders = random.sample(all_ids, len(all_ids))  # 11
        self.crane_pos = [3, 0]

        self.containers = {}
        for cid, data in self.containers_original.items():
            self.containers[cid] = {
                'cells': data['cells'].copy(),
                'size': data['size'],
                'name': data['name'],
                'pickup': data['pickup'],
                'priority': data['priority'],
                'color': data['color'],
                'picked': False,
                'delivered': False,
                'rotated': False
            }

        self.stacking_grid = [[[] for _ in range(self.stacking_width)] for _ in range(self.stacking_height)]
        self.steps = 0
        self.carrying = None
        self.done = False
        self.rotation_msg = ""
        self.target_pos = None
        self.next_pickup_target = None
        self.expected_priority = None
        self._update_targets()
        self._prev_distance = self._get_distance_to_current_target()
        return self._get_obs(), {}

    def _get_distance(self, pos, target):
        if target is None:
            return 0.0
        return abs(pos[0] - target[0]) + abs(pos[1] - target[1])

    def _get_current_target(self):
        if self.carrying is not None:
            return self.target_pos
        else:
            return self.next_pickup_target

    def _get_distance_to_current_target(self):
        target = self._get_current_target()
        if target is None:
            return 0.0
        return self._get_distance(self.crane_pos, target)

    def _get_obs(self):
        obs = np.zeros(self.state_size, dtype=np.float32)
        obs[0] = (self.crane_pos[0] - 3) / 3.0
        obs[1] = self.crane_pos[1] / (self.cols - 1)
        # picked status (11 adet)
        for i in range(11):
            obs[2 + i] = float(self.containers[i]['picked'])
        # stacking grid (16 hücre)
        idx = 2 + 11
        for r in range(self.stacking_height):
            for c in range(self.stacking_width):
                stack = self.stacking_grid[r][c]
                obs[idx] = stack[-1] / 10.0 if stack else 0.0  # max ID 10
                idx += 1
        # orders (11)
        for i in range(11):
            obs[idx + i] = 1.0 if i in self.orders else 0.0
        idx += 11
        obs[idx] = float(self.carrying is not None)
        idx += 1
        if self.target_pos is not None:
            obs[idx] = (self.target_pos[0] - 3) / 3.0
            obs[idx+1] = self.target_pos[1] / 3.0
            obs[idx+2] = self.target_pos[2] / 4.0
        idx += 3
        if self.next_pickup_target is not None:
            obs[idx] = self.next_pickup_target[0] / (self.rows - 1)
            obs[idx+1] = self.next_pickup_target[1] / (self.cols - 1)
        idx += 2
        if self.expected_priority is not None:
            obs[idx] = self.expected_priority / 2.0
        return obs

    def _is_blocked(self, pos):
        r, c = pos
        if r < 0 or r >= self.rows or c < 0 or c >= self.cols:
            return True
        if (r, c) in self.blocked_cells:
            return True
        return False  # konteynerler engel değil

    def _is_in_stacking_area(self, pos):
        r, c = pos
        return 3 <= r <= 6 and 0 <= c <= 3

    def _get_container_at(self, pos):
        pos_tuple = tuple(pos)
        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                if pos_tuple == data['pickup']:
                    return cid
        return None

    def _find_stacking_position(self, container_id, rotated=False):
        size = self.containers[container_id]['size']
        prio = self.containers[container_id]['priority']
        if rotated:
            size = (size[1], size[0])
        rows, cols = size
        if rows > self.stacking_height or cols > self.stacking_width:
            return None
        for level in range(5):  # max 4 kat
            for r in range(self.stacking_height - rows + 1):
                for c in range(self.stacking_width - cols + 1):
                    can_place = True
                    for i in range(rows):
                        for j in range(cols):
                            stack = self.stacking_grid[r + i][c + j]
                            if len(stack) > level:
                                can_place = False
                                break
                            if level > 0:
                                if len(stack) <= level - 1:
                                    can_place = False
                                    break
                                below_id = stack[level - 1]
                                below_prio = self.containers[below_id]['priority']
                                if prio <= below_prio:
                                    can_place = False
                                    break
                        if not can_place:
                            break
                    if can_place:
                        return (3 + r, c, level), size
        return None

    def _update_targets(self):
        if self.carrying is not None:
            pos_info = self._find_stacking_position(self.carrying, rotated=False)
            rotated = False
            if pos_info is None:
                pos_info = self._find_stacking_position(self.carrying, rotated=True)
                rotated = True
            if pos_info is not None:
                self.target_pos = pos_info[0]
                self.rotated_needed = rotated
            else:
                self.target_pos = None
                self.rotated_needed = False
        else:
            self.target_pos = None
            self.rotated_needed = False

        remaining = [oid for oid in self.orders
                     if not self.containers[oid]['picked'] and not self.containers[oid]['delivered']]
        if remaining:
            remaining.sort(key=lambda cid: (self.containers[cid]['priority'],
                                           self._get_distance(self.crane_pos, self.containers[cid]['pickup'])))
            next_id = remaining[0]
            self.next_pickup_target = self.containers[next_id]['pickup']
            self.expected_priority = self.containers[next_id]['priority']
        else:
            self.next_pickup_target = None
            self.expected_priority = None

    def step(self, action):
        self.steps += 1
        reward = -0.1
        self.rotation_msg = ""

        current_target = self._get_current_target()
        if current_target is not None:
            old_dist = self._prev_distance
            new_dist = self._get_distance(self.crane_pos, current_target)
            if new_dist < old_dist:
                reward += 0.3
            elif new_dist > old_dist:
                reward -= 0.3
            self._prev_distance = new_dist

        if self.carrying is None and self.next_pickup_target is not None:
            if tuple(self.crane_pos) == self.next_pickup_target and action != 4:
                reward -= 0.5

        if action == 0:  # Up
            new_pos = [self.crane_pos[0] - 1, self.crane_pos[1]]
            if not self._is_blocked(new_pos):
                self.crane_pos = new_pos
            else:
                reward -= 0.5
        elif action == 1:  # Down
            new_pos = [self.crane_pos[0] + 1, self.crane_pos[1]]
            if not self._is_blocked(new_pos):
                self.crane_pos = new_pos
            else:
                reward -= 0.5
        elif action == 2:  # Left
            new_pos = [self.crane_pos[0], self.crane_pos[1] - 1]
            if not self._is_blocked(new_pos):
                self.crane_pos = new_pos
            else:
                reward -= 0.5
        elif action == 3:  # Right
            new_pos = [self.crane_pos[0], self.crane_pos[1] + 1]
            if not self._is_blocked(new_pos):
                self.crane_pos = new_pos
            else:
                reward -= 0.5
        elif action == 4:  # Pickup
            if self.carrying is None:
                container_id = self._get_container_at(tuple(self.crane_pos))
                if container_id is not None and container_id in self.orders:
                    if self.containers[container_id]['priority'] == self.expected_priority:
                        self.containers[container_id]['picked'] = True
                        self.carrying = container_id
                        self._update_targets()
                        reward += 8.0
                        self._prev_distance = self._get_distance_to_current_target()
                    else:
                        reward -= 5.0
                else:
                    reward -= 2.0
            else:
                reward -= 2.0
        elif action == 5:  # Drop
            if self.carrying is not None:
                if not self._is_in_stacking_area(tuple(self.crane_pos)):
                    reward -= 4.0
                    self.rotation_msg = "Sadece istif alaninda birakabilirsiniz!"
                elif self.target_pos is None:
                    reward -= 4.0
                    self.rotation_msg = "Yerlestirilecek uygun yer yok!"
                elif (self.crane_pos[0], self.crane_pos[1]) != (self.target_pos[0], self.target_pos[1]):
                    reward -= 4.0
                    self.rotation_msg = f"Hedef: {self.target_pos[:2]}"
                else:
                    reward += 2.0
                    container_id = self.carrying
                    pos_info = self._find_stacking_position(container_id, rotated=False)
                    rotated = False
                    if pos_info is None:
                        pos_info = self._find_stacking_position(container_id, rotated=True)
                        rotated = True
                    if pos_info is not None and pos_info[0] == self.target_pos:
                        if rotated:
                            self.containers[container_id]['rotated'] = True
                            self.rotation_msg = f"Rotasyon: {self.containers[container_id]['name']}"
                        (r, c, level), size = pos_info
                        grid_r = r - 3
                        grid_c = c
                        for i in range(size[0]):
                            for j in range(size[1]):
                                stack = self.stacking_grid[grid_r + i][grid_c + j]
                                while len(stack) < level:
                                    stack.append(None)
                                stack.append(container_id)
                        self.containers[container_id]['delivered'] = True
                        self.carrying = None

                        delivered_count = sum(1 for oid in self.orders if self.containers[oid]['delivered'])
                        reward += 10.0 * delivered_count

                        self._update_targets()
                        self._prev_distance = self._get_distance_to_current_target()

                        if all(self.containers[oid]['delivered'] for oid in self.orders):
                            reward += 20.0
                            self.done = True
                    else:
                        reward -= 4.0
                        self.rotation_msg = "Rotasyonla bile sigmiyor!"
            else:
                reward -= 2.0

        if self.carrying is None and self.next_pickup_target is not None:
            if tuple(self.crane_pos) == self.next_pickup_target:
                reward += 1.0

        if self.steps >= self.max_steps:
            self.done = True

        return self._get_obs(), reward, self.done, False, {'rotation': self.rotation_msg}

    def render(self, ax=None):
        if ax is None:
            fig, ax = plt.subplots(figsize=(10, 10))

        grid = np.zeros((self.rows, self.cols), dtype=int)
        for cell in self.blocked_cells:
            grid[cell[0], cell[1]] = 1

        for r in range(self.stacking_height):
            for c in range(self.stacking_width):
                stack = self.stacking_grid[r][c]
                if stack:
                    top_id = stack[-1]
                    color = self.containers[top_id]['color']
                    if color == 'green':
                        grid[3 + r][c] = 3
                    elif color == 'yellow':
                        grid[3 + r][c] = 4
                    else:
                        grid[3 + r][c] = 5
                else:
                    grid[3 + r][c] = 2

        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                for cell in data['cells']:
                    color = data['color']
                    if color == 'green':
                        grid[cell[0]][cell[1]] = 3
                    elif color == 'yellow':
                        grid[cell[0]][cell[1]] = 4
                    else:
                        grid[cell[0]][cell[1]] = 5

        colors = ['white', 'gray', 'lightblue', 'green', 'gold', 'red']
        cmap = mcolors.ListedColormap(colors)
        ax.imshow(grid, cmap=cmap, vmin=0, vmax=5)

        for i in range(self.rows + 1):
            ax.axhline(i - 0.5, color='black', linewidth=0.5)
        for j in range(self.cols + 1):
            ax.axvline(j - 0.5, color='black', linewidth=0.5)

        for cid, data in self.containers.items():
            if not data['picked'] and not data['delivered']:
                cells = data['cells']
                center_r = sum(c[0] for c in cells) / len(cells)
                center_c = sum(c[1] for c in cells) / len(cells)
                ax.text(center_c, center_r, data['name'], ha='center', va='center',
                       fontsize=6, fontweight='bold')

        for r in range(self.stacking_height):
            for c in range(self.stacking_width):
                stack = self.stacking_grid[r][c]
                if stack:
                    top_id = stack[-1]
                    level = len(stack) - 1
                    label = f'K{top_id}'
                    if level > 0:
                        label += f'\nL{level}'
                    ax.text(c, 3 + r, label, ha='center', va='center', fontsize=6, fontweight='bold')

        if self.target_pos is not None:
            tr, tc, tl = self.target_pos
            rect = Rectangle((tc - 0.5, tr - 0.5), 1, 1, linewidth=2, edgecolor='yellow', facecolor='none', linestyle='--')
            ax.add_patch(rect)

        if self.next_pickup_target is not None and self.carrying is None:
            pr, pc = self.next_pickup_target
            edge_color = 'green' if self.expected_priority == 0 else ('yellow' if self.expected_priority == 1 else 'red')
            rect = Rectangle((pc - 0.5, pr - 0.5), 1, 1, linewidth=2, edgecolor=edge_color, facecolor='none', linestyle='--')
            ax.add_patch(rect)

        ax.plot(self.crane_pos[1], self.crane_pos[0], 'r*', markersize=20,
               markeredgecolor='black', markeredgewidth=2)

        if self.carrying is not None:
            ax.text(self.crane_pos[1], self.crane_pos[0] - 0.4,
                   f'[{self.containers[self.carrying]["name"]}]',
                   ha='center', va='center', fontsize=8, color='red', fontweight='bold',
                   bbox=dict(boxstyle='round,pad=0.2', facecolor='white', alpha=0.8))

        ax.set_xticks([])
        ax.set_yticks([])
        ax.set_title(f'Siparis: {len(self.orders)} konteyner | Tasinan: {self.containers[self.carrying]["name"] if self.carrying else "Yok"}', fontsize=10)
        return ax

# ============================================
# DQN AGENT (optimize)
# ============================================
class DQN(nn.Module):
    def __init__(self, state_size, action_size):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(state_size, 512),
            nn.ReLU(),
            nn.Linear(512, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, action_size)
        )
    def forward(self, x):
        return self.network(x)

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=10000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.9995
        self.learning_rate = 0.0003
        self.batch_size = 128
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        self.model = DQN(state_size, action_size).to(self.device)
        self.target_model = DQN(state_size, action_size).to(self.device)
        self.update_target_model()
        self.optimizer = optim.Adam(self.model.parameters(), lr=self.learning_rate)
        self.criterion = nn.MSELoss()

    def update_target_model(self):
        self.target_model.load_state_dict(self.model.state_dict())

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state, training=True):
        if training and np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        state_tensor = torch.FloatTensor(state).unsqueeze(0).to(self.device)
        with torch.no_grad():
            q_values = self.model(state_tensor)
        return q_values.cpu().numpy()[0].argmax()

    def replay(self):
        if len(self.memory) < self.batch_size:
            return
        minibatch = random.sample(self.memory, self.batch_size)
        states = torch.FloatTensor([t[0] for t in minibatch]).to(self.device)
        actions = torch.LongTensor([t[1] for t in minibatch]).to(self.device)
        rewards = torch.FloatTensor([t[2] for t in minibatch]).to(self.device)
        next_states = torch.FloatTensor([t[3] for t in minibatch]).to(self.device)
        dones = torch.FloatTensor([t[4] for t in minibatch]).to(self.device)
        current_q = self.model(states).gather(1, actions.unsqueeze(1))
        with torch.no_grad():
            max_next_q = self.target_model(next_states).max(1)[0]
            target_q = rewards + (1 - dones) * self.gamma * max_next_q
        loss = self.criterion(current_q.squeeze(), target_q)
        self.optimizer.zero_grad()
        loss.backward()
        self.optimizer.step()
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# ============================================
# EĞİTİM
# ============================================
print("Ortam olusturuluyor...")
env = ContainerYardEnv()
state_size = env.state_size
action_size = 6
agent = DQNAgent(state_size, action_size)

episodes = 2000
print(f"Egitim basliyor... ({episodes} bolum)")

rewards_history = []
for e in tqdm(range(episodes), desc="Egitim"):
    state, _ = env.reset()
    total_reward = 0
    while True:
        action = agent.act(state)
        next_state, reward, done, _, _ = env.step(action)
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        total_reward += reward
        if done:
            agent.update_target_model()
            break
        if len(agent.memory) > agent.batch_size:
            agent.replay()
    rewards_history.append(total_reward)
    if (e + 1) % 200 == 0:
        avg_reward = np.mean(rewards_history[-200:])
        tqdm.write(f"Bolum: {e+1}/{episodes}, Ort. Odul: {avg_reward:.2f}, Epsilon: {agent.epsilon:.3f}")

print("Egitim tamamlandi!")

plt.figure(figsize=(10, 4))
plt.plot(rewards_history)
plt.xlabel('Bolum')
plt.ylabel('Toplam Odul')
plt.title('Egitim Sureci')
plt.grid(True)
plt.show()

# ============================================
# TEST VE GIF
# ============================================
print("Test basliyor...")
test_env = ContainerYardEnv()
state, _ = test_env.reset()
orders_str = ", ".join([f"{test_env.containers[oid]['name']}" for oid in test_env.orders])
print(f"Siparis edilen (11 adet): {orders_str}")

frames = []
fig, ax = plt.subplots(figsize=(10, 10))
step = 0
max_test_steps = 1000

while step < max_test_steps:
    ax.clear()
    test_env.render(ax)
    fig.canvas.draw()
    img_array = np.array(fig.canvas.renderer.buffer_rgba())
    frames.append(img_array[:, :, :3])
    action = agent.act(state, training=False)
    state, reward, done, _, _ = test_env.step(action)
    step += 1
    if done:
        ax.clear()
        test_env.render(ax)
        fig.canvas.draw()
        img_array = np.array(fig.canvas.renderer.buffer_rgba())
        frames.append(img_array[:, :, :3])
        break

plt.close()
gif_path = 'container_yard_11stack.gif'
imageio.mimsave(gif_path, frames, fps=5, loop=0)
print(f"Test tamamlandi ({step} adim), GIF: {gif_path}")
from IPython.display import Image as IPImage
IPImage(filename=gif_path)